In [1]:
import glob
import pandas as pd
pd.set_option('display.max_columns', None)

In [2]:
def load(pattern):
    paths = glob.glob(pattern)
    if not paths:
        raise FileNotFoundError(f"There is no file: {pattern}")
    return pd.read_csv(paths[0])

lab = load('Train-*.csv')
ben = load('Train_Beneficiarydata*.csv')
ip  = load('Train_Inpatientdata*.csv')
op  = load('Train_Outpatientdata*.csv')

for name, d in [('labels', lab), ('beneficiary', ben), ('inpatient', ip), ('outpatient', op)]:
    print(name, d.shape)

labels (5410, 2)
beneficiary (138556, 25)
inpatient (40474, 30)
outpatient (517737, 27)


In [3]:
lab.head()

,Provider,PotentialFraud
0,PRV51001,No
1,PRV51003,Yes
2,PRV51004,No
3,PRV51005,Yes
4,PRV51007,No


In [4]:
ip['Type'] = 'IP'
op['Type'] = 'OP'
claims = pd.concat([ip, op], ignore_index=True)
claims = claims.merge(ben, on='BeneID', how='left')
claims = claims.merge(lab, on='Provider', how='left')
claims['Fraud'] = (claims['PotentialFraud'] == 'Yes').astype(int)
claims.shape

(558211, 57)

In [5]:
claims['AdmissionDt'] = pd.to_datetime(claims['AdmissionDt'])
claims['DischargeDt'] = pd.to_datetime(claims['DischargeDt'])
claims['Days'] = (claims['DischargeDt'] - claims['AdmissionDt']).dt.days + 1
claims['Cost_Per_Day'] = claims['InscClaimAmtReimbursed'] / claims['Days']
claims[['Provider', 'InscClaimAmtReimbursed', 'Days', 'Cost_Per_Day', 'Fraud']].head()

,Provider,InscClaimAmtReimbursed,Days,Cost_Per_Day,Fraud
0,PRV55912,26000,7.0,3714.285714,1
1,PRV55907,5000,3.0,1666.666667,0
2,PRV56046,5000,4.0,1250.000000,0
3,PRV52405,5000,9.0,555.555556,0
4,PRV56614,10000,18.0,555.555556,0


In [6]:
prov = claims.groupby('Provider').agg(
    n_claims=('ClaimID', 'count'),
    n_patients=('BeneID', 'nunique'),
    avg_claim=('InscClaimAmtReimbursed', 'mean'),
    total_claim=('InscClaimAmtReimbursed', 'sum'),
    ip_share=('Type', lambda s: (s == 'IP').mean()),
    avg_days=('Days', 'mean'),
    avg_cost_per_day=('Cost_Per_Day', 'mean'),
    n_doctors=('AttendingPhysician', 'nunique'),
    Fraud=('Fraud', 'max'),
).reset_index()

print(prov.shape)
print(prov['Fraud'].value_counts(normalize=True))
prov.head()

(5410, 10)
Fraud
0    0.90647
1    0.09353
Name: proportion, dtype: float64


,Provider,n_claims,n_patients,avg_claim,total_claim,ip_share,avg_days,avg_cost_per_day,n_doctors,Fraud
0,PRV51001,25,24,4185.600000,104640,0.200000,6.000000,3700.000000,14,0
1,PRV51003,132,117,4588.409091,605670,0.469697,6.161290,2137.423195,44,1
2,PRV51004,149,138,350.134228,52170,0.000000,NaN,NaN,38,0
3,PRV51005,1165,495,241.124464,280910,0.000000,NaN,NaN,6,1
4,PRV51007,72,58,468.194444,33710,0.041667,6.333333,1080.555556,10,0


In [7]:
claims.to_csv('claims_merged.csv', index=False)
prov.to_csv('provider_table.csv', index=False)

In [8]:
claims.shape

(558211, 59)

In [9]:
prov.shape

(5410, 10)

In [10]:
use = ['Provider','BeneID','ClaimID','InscClaimAmtReimbursed','DeductibleAmtPaid','Type',
       'AttendingPhysician','OperatingPhysician','ClmDiagnosisCode_1','ClmProcedureCode_1',
       'ChronicCond_Diabetes','ChronicCond_Heartfailure','Fraud']
head = pd.read_csv('claims_merged.csv', nrows=1)
use = [c for c in use if c in head.columns]
claims = pd.read_csv('claims_merged.csv', usecols=use)
print(claims.shape)
claims.head(3)

(558211, 13)


,BeneID,ClaimID,Provider,InscClaimAmtReimbursed,AttendingPhysician,OperatingPhysician,DeductibleAmtPaid,ClmDiagnosisCode_1,ClmProcedureCode_1,Type,ChronicCond_Heartfailure,ChronicCond_Diabetes,Fraud
0,BENE11001,CLM46614,PRV55912,26000,PHY390922,NaN,1068.0,1970,NaN,IP,2,1,1
1,BENE11001,CLM66048,PRV55907,5000,PHY318495,PHY318495,1068.0,6186,7092.0,IP,2,1,0
2,BENE11001,CLM68358,PRV56046,5000,PHY372395,NaN,1068.0,29623,NaN,IP,2,1,0


In [11]:
prov.isna().sum()

Provider               0
n_claims               0
n_patients             0
avg_claim              0
total_claim            0
ip_share               0
avg_days            3318
avg_cost_per_day    3318
n_doctors              0
Fraud                  0
dtype: int64

In [12]:
prov.isna().sum().sum()

np.int64(6636)

In [13]:
prov.isna().sum()[prov.isna().sum() > 0]

avg_days            3318
avg_cost_per_day    3318
dtype: int64

In [14]:
prov.info()

<class 'pandas.DataFrame'>
RangeIndex: 5410 entries, 0 to 5409
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Provider          5410 non-null   str    
 1   n_claims          5410 non-null   int64  
 2   n_patients        5410 non-null   int64  
 3   avg_claim         5410 non-null   float64
 4   total_claim       5410 non-null   int64  
 5   ip_share          5410 non-null   float64
 6   avg_days          2092 non-null   float64
 7   avg_cost_per_day  2092 non-null   float64
 8   n_doctors         5410 non-null   int64  
 9   Fraud             5410 non-null   int64  
dtypes: float64(4), int64(5), str(1)
memory usage: 465.7 KB


In [15]:
claims.isna().sum()[claims.isna().sum() > 0]

AttendingPhysician      1508
OperatingPhysician    443764
DeductibleAmtPaid        899
ClmDiagnosisCode_1     10453
ClmProcedureCode_1    534901
dtype: int64